## **Section B: RAG Fundamentals, Implementation**

**Corpus : Nusrat Fateh Ali Khan**

In [1]:
corpus = """
Nusrat Fateh Ali Khan (NFAK) was a Pakistani Qawwali singer, musician, and composer who became one of the most influential voices in South Asian music.
He is mainly known for his powerful vocals and his contribution to Qawwali, a traditional form of Sufi devotional music.
His distinctive singing style and ability to perform for long durations made him a highly respected figure in international music.

Nusrat Fateh Ali Khan came from a family with a long tradition of Qawwali music.
He learned classical and traditional musical techniques from his family and gradually developed his own unique style.
His performances combined traditional Qawwali with elements of classical music, allowing his music to reach audiences beyond South Asia.

Nusrat Fateh Ali Khan became particularly famous for his energetic live performances.
His songs often featured powerful vocals, rhythmic clapping, harmonium, and a group of supporting singers.
His ability to build musical intensity during performances made him one of the most celebrated Qawwali performers of his generation.

Nusrat also contributed to international music and worked with musicians and producers outside South Asia.
He collaborated with artists such as Peter Gabriel and contributed to projects that introduced Qawwali and South Asian devotional music to wider global audiences.
His international work helped establish Qawwali as a recognized form of world music.

Nusrat Fateh Ali Khan received several awards and international recognitions during his career.
He was associated with numerous successful recordings and performed in many countries around the world.
His contribution to music made him an important cultural figure and helped influence musicians in South Asia and internationally.
"""

In [8]:
pip install sentence-transformers  rank_bm25 numpy scikit-learn


### **CHUNKING**

In [9]:
import re

def fixed_size_chunk(text, chunk_size=300, overlap=50):
    """Split text into fixed-size chunks with overlap between consecutive chunks."""
    chunks = []
    start = 0
    text = re.sub(r"\s+", " ", text).strip()  # normalize whitespace
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap  # move forward, keeping overlap
    return chunks

CHUNK_SIZE = 300
OVERLAP = 50

chunks = fixed_size_chunk(corpus, CHUNK_SIZE, OVERLAP)

print(f"Number of chunks: {len(chunks)}\n")
for i, c in enumerate(chunks):
    print(f"--- Chunk {i} ({len(c)} chars) ---")
    print(c)
    print()

Number of chunks: 7

--- Chunk 0 (300 chars) ---
Nusrat Fateh Ali Khan (NFAK) was a Pakistani Qawwali singer, musician, and composer who became one of the most influential voices in South Asian music. He is mainly known for his powerful vocals and his contribution to Qawwali, a traditional form of Sufi devotional music. His distinctive singing sty

--- Chunk 1 (299 chars) ---
Sufi devotional music. His distinctive singing style and ability to perform for long durations made him a highly respected figure in international music. Nusrat Fateh Ali Khan came from a family with a long tradition of Qawwali music. He learned classical and traditional musical techniques from his

--- Chunk 2 (300 chars) ---
sical and traditional musical techniques from his family and gradually developed his own unique style. His performances combined traditional Qawwali with elements of classical music, allowing his music to reach audiences beyond South Asia. Nusrat Fateh Ali Khan became particularly famous for

**Explanation:**                                
*  The corpus is split into overlapping windows of fixed character length rather
than by sentence or paragraph
* this is simple and works on any text without needing sentence
detection.
* The 50-character overlap means the tail of one chunk reappears at the start of the
next, so a sentence that straddles a boundary still appears whole in at least one chunk.
*  The
trade-off: fixed-size chunking can still cut a sentence awkwardly (it doesn't respect sentence
boundaries), but it's fast, predictable, and good enough for a baseline RAG pipeline.

## **Dense Retrieval**

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def dense_retrieve(query, chunks, chunk_embeddings, top_k=3):
    """Embed the query and return the top_k most similar chunks by cosine similarity."""
    query_embedding = model.encode([query], convert_to_numpy=True)
    sims = cosine_similarity(query_embedding, chunk_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(idx, sims[idx], chunks[idx]) for idx in top_idx]

# --- Test with 2 sample queries ---
queries = [
    "What made Nusrat Fateh Ali Khan famous in Qawwali and international music?",
    "How did Nusrat Fateh Ali Khan contribute to the global popularity of Qawwali?",
]

dense_results = {}
for q in queries:
    print(f"\n=== Dense retrieval | Query: '{q}' ===")
    results = dense_retrieve(q, chunks, chunk_embeddings, top_k=3)
    dense_results[q] = results
    for rank, (idx, score, chunk) in enumerate(results, start=1):
        print(f"\nRank {rank} | Chunk {idx} | cosine similarity = {score:.4f}")
        print(chunk)


=== Dense retrieval | Query: 'What made Nusrat Fateh Ali Khan famous in Qawwali and international music?' ===

Rank 1 | Chunk 5 | cosine similarity = 0.8442
s that introduced Qawwali and South Asian devotional music to wider global audiences. His international work helped establish Qawwali as a recognized form of world music. Nusrat Fateh Ali Khan received several awards and international recognitions during his career. He was associated with numerous s

Rank 2 | Chunk 2 | cosine similarity = 0.8315
sical and traditional musical techniques from his family and gradually developed his own unique style. His performances combined traditional Qawwali with elements of classical music, allowing his music to reach audiences beyond South Asia. Nusrat Fateh Ali Khan became particularly famous for his ene

Rank 3 | Chunk 1 | cosine similarity = 0.8131
Sufi devotional music. His distinctive singing style and ability to perform for long durations made him a highly respected figure in international

**Explanation:**                                
* Each chunk and each query gets converted into a dense vector (embedding) that
captures its *meaning*, not just its exact words.  
 * Cosine similarity measures the angle between
the query vector and each chunk vector
* Higher score means the chunk is more semantically
related to the query, even if it doesn't share the same wording.
* This is why dense retrieval can
find relevant chunks that use synonyms or paraphrases of the query terms, which keyword search
would miss.


## **Sparse retrieval (BM25)**

In [17]:
from rank_bm25 import BM25Okapi

def simple_tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenized_chunks = [simple_tokenize(c) for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

def bm25_retrieve(query, chunks, bm25, top_k=3):
    tokenized_query = simple_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(idx, scores[idx], chunks[idx]) for idx in top_idx]

bm25_results = {}
for q in queries:
    print(f"\n=== BM25 retrieval | Query: '{q}' ===")
    results = bm25_retrieve(q, chunks, bm25, top_k=3)
    bm25_results[q] = results
    for rank, (idx, score, chunk) in enumerate(results, start=1):
        print(f"\nRank {rank} | Chunk {idx} | BM25 score = {score:.4f}")
        print(chunk)


=== BM25 retrieval | Query: 'What made Nusrat Fateh Ali Khan famous in Qawwali and international music?' ===

Rank 1 | Chunk 2 | BM25 score = 2.5755
sical and traditional musical techniques from his family and gradually developed his own unique style. His performances combined traditional Qawwali with elements of classical music, allowing his music to reach audiences beyond South Asia. Nusrat Fateh Ali Khan became particularly famous for his ene

Rank 2 | Chunk 1 | BM25 score = 2.5254
Sufi devotional music. His distinctive singing style and ability to perform for long durations made him a highly respected figure in international music. Nusrat Fateh Ali Khan came from a family with a long tradition of Qawwali music. He learned classical and traditional musical techniques from his

Rank 3 | Chunk 5 | BM25 score = 2.2229
s that introduced Qawwali and South Asian devotional music to wider global audiences. His international work helped establish Qawwali as a recognized form of world music

**Explanation:**
* BM25 is a keyword-based scoring method — it ranks chunks by how often the
query's exact terms appear in them, weighted by how rare/informative those terms are across the
whole corpus (and adjusted for chunk length).
 * Unlike dense retrieval, it has no notion of
meaning: a chunk only scores well if it literally contains the query's words.
* This makes it
strong for queries with specific names, numbers, or jargon, but weak when the query is phrased
differently from the source text.

### **Comparison: Dense vs BM25**

For each query, compare the top chunk(s) returned by dense retrieval vs BM25:

- **Query 1:** *(fill in after running — e.g. "Dense retrieval worked better because the query
  used different wording than the source text, and embeddings capture semantic/paraphrase
  similarity that exact keyword matching (BM25) misses.")*
- **Query 2:** *(fill in after running — e.g. "BM25 worked better because the query contained a
  specific keyword/name/number that appears verbatim in the corpus, and exact term matching
  ranks that chunk highly, whereas dense embeddings can dilute rare/specific terms into a more
  generic semantic representation.")*

**General takeaway:** BM25 tends to win on queries with specific keywords, exact terms, names,
or numbers. Dense retrieval tends to win on queries that are paraphrased or semantically related
to the text without sharing exact vocabulary.